# TerraMind Embedding Comparison
Compare embeddings from PhiSat-2 triplets (real, simulated, Sentinel-2) using cosine similarity.

## 1. Import Libraries

In [ ]:
import torch
import torch.nn.functional as F
import h5py
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

: 

## 2. Load TerraMind Model

In [ ]:
from terratorch.tasks import SemanticSegmentationTask
from terra_sat_drift.data_simulation.phisat2_constants import S2_BANDS_NAMES, S2_BANDS

# Load pre-trained TerraMind model
# print("Loading pre-trained TerraMind model...")
# checkpoint_path = Path("/shared/home/elucas/terra-sat-drift/outputs/terramind_sen1floods_simulated/checkpoints_stage1/best-val_mIoU.ckpt")

# if checkpoint_path.exists():
#     model = SemanticSegmentationTask.load_from_checkpoint(str(checkpoint_path))
#     print(f"✓ Model loaded from checkpoint: {checkpoint_path}")
# else:
#     print(f"⚠ Checkpoint not found, creating fresh model...")
    # Create a fresh model if checkpoint doesn't exist
from terratorch import BACKBONE_REGISTRY
    
backbone_name = "terramind_v1_small"
model_args = {
    "backbone": backbone_name,
    "backbone_pretrained": True,
    "backbone_modalities": ["S2L1C"],
    "backbone_bands": {"S2L1C": S2_BANDS},
    "decoder": "IdentityDecoder",
    "decoder_channels": [256, 128, 64, 32],
    "necks": [
        {"name": "SelectIndices", "indices": [1, 3, 4, 5]},
        {"name": "ReshapeTokensToImage", "remove_cls_token": False},
        {"name": "LearnedInterpolateToPyramidal"},
    ],
        "num_classes": 2,
}
model = SemanticSegmentationTask(
    model_factory="EncoderDecoderFactory",
    model_args=model_args,
    lr=1e-4,
)
print("model created with pretrained backbone")

model.to(device)
model.eval()
print("Model ready for inference")

✓ Fresh model created with pretrained backbone
Model ready for inference


In [ ]:
from terratorch.models.backbones.terramind.model.terramind_vit import TerraMindViT

model = TerraMindViT(
    img_size=256,
    pretrained=True,
    modalities=["S2L1C"],
    bands={"S2L1C": S2_BANDS},
).to(device)
print("✓ TerraMindViT backbone loaded and moved to device")

## 3. Load PhiSat-2 Triplets

In [3]:
h5_path = Path("/shared/projects/phisat2/data/processed/triplets_v1/phisat2_s2b_dataset_v1.h5")

# Open dataset and select random samples
f = h5py.File(h5_path, 'r')
n_samples = f['real/images'].shape[0]
sample_indices = np.random.choice(n_samples, size=3, replace=False)

print(f"Loaded PhiSat-2 dataset: {n_samples} total samples")
print(f"Selected samples: {sample_indices}")

# Extract triplets - we'll select S2 bands that match TerraMind's 7-band input
# TerraMind expects: B2, B3, B4, B5, B6, B7, B8 (7 bands from Sentinel-2)
# Sentinel-2 dataset order: B02, B03, B04, B05, B06, B07, B08
# We need to map: sim (8 bands) -> 7 bands to match S2 ordering

triplets = {'real': [], 'sim': [], 's2b': []}

for idx in sample_indices:
    # Load PhiSat-2 real (8 bands: PAN + 7 optical)
    # Use bands 1-7 (skip PAN at index 0) to get 7 bands
    real_data = f['real/images'][idx][1:8]  # (7, 256, 256)
    triplets['real'].append(torch.from_numpy(real_data).float())
    
    # Load PhiSat-2 simulated (8 bands: similar to real)
    sim_data = f['sim/images'][idx][1:8]  # (7, 256, 256)
    triplets['sim'].append(torch.from_numpy(sim_data).float())
    
    # Load Sentinel-2B (7 bands: B02-B08)
    s2b_data = f['s2b/images'][idx]  # (7, 256, 256)
    triplets['s2b'].append(torch.from_numpy(s2b_data).float())

print(f"✓ Loaded {len(sample_indices)} triplets, each with shape (7, 256, 256)")

Loaded PhiSat-2 dataset: 259150 total samples
Selected samples: [ 45486 215753  13249]
✓ Loaded 3 triplets, each with shape (7, 256, 256)


## 4. Extract Embeddings from Backbone

In [ ]:
def extract_embeddings(model: SemanticSegmentationTask, images_list: list, device: torch.device):
    """Extract embeddings from model backbone."""
    embeddings = []
    
    with torch.no_grad():
        for img_tensor in images_list:
            # Add batch dimension and send to device
            img_batch = img_tensor.unsqueeze(0).to(device)  # (1, 7, 256, 256)
            
            # Get model output
            output = model.predict_step(img_batch)
            
            # Extract embeddings from the backbone
            # The output contains the segmentation logits, but we want the encoded features
            # For simplicity, we'll use the full output and flatten it
            embedding = output.flatten().cpu().numpy()
            embeddings.append(embedding)
    
    return np.array(embeddings)

# Extract embeddings for all modalities
print("Extracting embeddings from backbone...")
embeddings_real = extract_embeddings(model, triplets['real'], device)
embeddings_sim = extract_embeddings(model, triplets['sim'], device)
embeddings_s2b = extract_embeddings(model, triplets['s2b'], device)

print(f"✓ Real embeddings shape: {embeddings_real.shape}")
print(f"✓ Sim embeddings shape: {embeddings_sim.shape}")
print(f"✓ S2B embeddings shape: {embeddings_s2b.shape}")

Extracting embeddings from backbone...


AttributeError: 'ModelOutput' object has no attribute 'flatten'

## 5. Compute Cosine Similarity

In [ ]:
# Compute pairwise cosine similarities
print("Computing cosine similarities...")

similarities = {}
for i in range(len(sample_indices)):
    # Real vs Simulated
    sim_real_sim = cosine_similarity(
        embeddings_real[i].reshape(1, -1),
        embeddings_sim[i].reshape(1, -1)
    )[0, 0]
    
    # Real vs S2B
    sim_real_s2b = cosine_similarity(
        embeddings_real[i].reshape(1, -1),
        embeddings_s2b[i].reshape(1, -1)
    )[0, 0]
    
    # Simulated vs S2B
    sim_sim_s2b = cosine_similarity(
        embeddings_sim[i].reshape(1, -1),
        embeddings_s2b[i].reshape(1, -1)
    )[0, 0]
    
    similarities[i] = {
        'real-sim': sim_real_sim,
        'real-s2b': sim_real_s2b,
        'sim-s2b': sim_sim_s2b
    }
    
    print(f"\nSample {sample_indices[i]}:")
    print(f"  Real vs Simulated: {sim_real_sim:.4f}")
    print(f"  Real vs S2B:       {sim_real_s2b:.4f}")
    print(f"  Simulated vs S2B:  {sim_sim_s2b:.4f}")

## 6. Visualize Results

In [ ]:
# Create similarity matrix visualization
fig, axes = plt.subplots(1, len(sample_indices), figsize=(5*len(sample_indices), 4))

if len(sample_indices) == 1:
    axes = [axes]

similarity_pairs = ['real-sim', 'real-s2b', 'sim-s2b']

for idx, sample_idx in enumerate(sample_indices):
    # Create matrix for heatmap
    matrix = np.array([
        [1.0, similarities[idx]['real-sim'], similarities[idx]['real-s2b']],
        [similarities[idx]['real-sim'], 1.0, similarities[idx]['sim-s2b']],
        [similarities[idx]['real-s2b'], similarities[idx]['sim-s2b'], 1.0]
    ])
    
    # Plot heatmap
    sns.heatmap(matrix, annot=True, fmt='.4f', cmap='coolwarm', 
                xticklabels=['Real', 'Sim', 'S2B'],
                yticklabels=['Real', 'Sim', 'S2B'],
                vmin=0, vmax=1, ax=axes[idx], cbar=True)
    axes[idx].set_title(f'Sample {sample_idx}\nCosine Similarity')

plt.tight_layout()
plt.show()

# Plot bar chart
fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(similarities))
width = 0.25

real_sim_vals = [similarities[i]['real-sim'] for i in range(len(sample_indices))]
real_s2b_vals = [similarities[i]['real-s2b'] for i in range(len(sample_indices))]
sim_s2b_vals = [similarities[i]['sim-s2b'] for i in range(len(sample_indices))]

ax.bar(x - width, real_sim_vals, width, label='Real vs Simulated', alpha=0.8)
ax.bar(x, real_s2b_vals, width, label='Real vs S2B', alpha=0.8)
ax.bar(x + width, sim_s2b_vals, width, label='Simulated vs S2B', alpha=0.8)

ax.set_ylabel('Cosine Similarity', fontsize=12)
ax.set_xlabel('Sample Index', fontsize=12)
ax.set_title('Embedding Similarity Across Triplets', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels([f'S{idx}' for idx in sample_indices])
ax.legend()
ax.set_ylim([0, 1])
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Visualization complete")

## 7. Summary Statistics

In [ ]:
# Summary statistics
all_real_sim = [similarities[i]['real-sim'] for i in range(len(sample_indices))]
all_real_s2b = [similarities[i]['real-s2b'] for i in range(len(sample_indices))]
all_sim_s2b = [similarities[i]['sim-s2b'] for i in range(len(sample_indices))]

print("\n" + "="*60)
print("EMBEDDING SIMILARITY SUMMARY")
print("="*60)

print(f"\nReal vs Simulated:")
print(f"  Mean: {np.mean(all_real_sim):.4f} ± {np.std(all_real_sim):.4f}")
print(f"  Min/Max: {np.min(all_real_sim):.4f} / {np.max(all_real_sim):.4f}")

print(f"\nReal vs S2B:")
print(f"  Mean: {np.mean(all_real_s2b):.4f} ± {np.std(all_real_s2b):.4f}")
print(f"  Min/Max: {np.min(all_real_s2b):.4f} / {np.max(all_real_s2b):.4f}")

print(f"\nSimulated vs S2B:")
print(f"  Mean: {np.mean(all_sim_s2b):.4f} ± {np.std(all_sim_s2b):.4f}")
print(f"  Min/Max: {np.min(all_sim_s2b):.4f} / {np.max(all_sim_s2b):.4f}")

print("\n" + "="*60)

# Close dataset
f.close()
print("✓ HDF5 file closed")